<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/notebooks/01_importar_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Importação de Dados do Kaggle para o BigQuery
Dataset: Mercado de Games

Este notebook utiliza a biblioteca `kagglehub` para baixar os arquivos originais (.csv) do Kaggle em tempo real e em seguida enviá-los diretamente para o Google BigQuery.

## 1. Importação das Bibliotecas

In [ ]:
import pandas as pd
import os
import kagglehub
from google.cloud import bigquery
from google.colab import auth

# Autenticação no Google Colab
auth.authenticate_user()

## 2. Autenticação no BigQuery

In [ ]:
project_id = 'directed-mender-489100-m3'
dataset_id = 'games_data'
client = bigquery.Client(project=project_id)

## 3. Baixar os Dados Diretamente do Kaggle

In [ ]:
print("Iniciando o download da base de dados mais recente do Kaggle...")

data_path = kagglehub.dataset_download("gabrigoncam/video-game-dataset-multidimensional")

print("Arquivos baixados na pasta local:", data_path)

all_files = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

print(f"\nEncontrados {len(all_files)} arquivos .csv prontos para envio ao BigQuery.")

## 4. Subir Tudo para o BigQuery

In [ ]:
for file_path in all_files:
    file_name = os.path.basename(file_path)
    table_name = file_name.replace('.csv', '')

    table_id = f"{project_id}.{dataset_id}.{table_name}"

    print(f"Enviando [{file_name}] para a tabela [{table_id}]...")

    if file_name == 'games.csv':
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    elif file_name == 'game_platforms.csv':
        df = pd.read_csv(file_path, dtype={'game_id': str})
    elif file_name == 'game_developers.csv':
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    else:
        df = pd.read_csv(file_path)

    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()

    print(f"\t>> ✅ Sucesso! Tabela {table_name} carregada.\n")

print("Todos os dados foram enviados para o Google BigQuery com sucesso!")